# Quick EDA：科学技術プロジェクト二値分類

baseline作成前に確認する基本EDAです。表を眺めるだけでなく、target、年度、欠損、数値、カテゴリ、日本語テキスト、`project_name`重複、train/test差、時系列CVをグラフ中心で確認します。

このNotebookでは特徴量生成やモデル学習を行いません。`project_start_year == -1`も元データから削除せず、年度グラフとCVでだけ異常値として除外します。

In [ ]:
from pathlib import Path
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp

try:
    import japanize_matplotlib
except ImportError:
    japanize_matplotlib = None
    warnings.warn('Install requirements.txt to render Japanese labels correctly.')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from validation import encode_binary_target, make_time_series_cv

sns.set_theme(style='whitegrid', context='notebook')
if japanize_matplotlib is not None:
    japanize_matplotlib.japanize()
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## 1. 設定とデータ読込

列名は配布データに合わせてここだけ編集します。自動検出された数値・カテゴリ列は、読込後に必ず表示して確認します。`PLOT_SAMPLE_SIZE`は描画だけに使い、集計値は原則として全行から計算します。

In [ ]:
TARGET_COL = 'science_tech_decision'
ID_COL = 'project_id'
YEAR_COL = 'project_start_year'
PROJECT_COL = 'project_name'
TEXT_COLS = ['project_name', 'project_objective', 'project_summary']
MANUAL_NUMERIC_COLS = ['project_start_year', 'project_fiscal_year', 'project_end_year', 'budget']
MANUAL_CATEGORICAL_COLS = ['responsible_ministry']

TOP_N = 15
MIN_CATEGORY_COUNT = 20
PLOT_SAMPLE_SIZE = 50_000
MAX_AUTO_CATEGORIES = 100
SAVE_FIGURES = False
FIGURE_DIR = PROJECT_ROOT / 'data' / 'eda_figures'

train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')

required_train = [TARGET_COL, ID_COL, YEAR_COL, PROJECT_COL]
required_test = [ID_COL, YEAR_COL, PROJECT_COL]
missing_train = [column for column in required_train if column not in train.columns]
missing_test = [column for column in required_test if column not in test.columns]
if missing_train or missing_test:
    raise KeyError(f'Missing columns: train={missing_train}, test={missing_test}')
train[TARGET_COL] = encode_binary_target(train[TARGET_COL])  # 該当=1, 非該当=0
if not train.index.is_unique or not test.index.is_unique:
    raise ValueError('DataFrame index must be unique.')

common_cols = [column for column in train.columns if column in test.columns]
text_cols = [column for column in TEXT_COLS if column in common_cols]
auto_numeric = [
    column for column in common_cols
    if pd.api.types.is_numeric_dtype(train[column]) and column not in {TARGET_COL, ID_COL}
]
numeric_cols = list(dict.fromkeys(
    [column for column in MANUAL_NUMERIC_COLS if column in common_cols] + auto_numeric
))
auto_categorical = [
    column for column in common_cols
    if column not in {ID_COL, *text_cols}
    and not pd.api.types.is_numeric_dtype(train[column])
    and train[column].nunique(dropna=True) <= MAX_AUTO_CATEGORIES
]
categorical_cols = list(dict.fromkeys(
    [column for column in MANUAL_CATEGORICAL_COLS if column in common_cols] + auto_categorical
))

def sampled(frame, n=PLOT_SAMPLE_SIZE, random_state=42):
    return frame if len(frame) <= n else frame.sample(n=n, random_state=random_state)

def finish_figure(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()

print('train:', train.shape, 'test:', test.shape)
print('numeric columns    :', numeric_cols)
print('categorical columns:', categorical_cols)
print('text columns       :', text_cols)

## 2. 全体像・スキーマ

行列数、メモリ、重複、ID、列型、train/testの列差を最初に確認します。IDは特徴量として使わず、submissionの行対応に使います。

In [ ]:
def year_range(frame):
    years = pd.to_numeric(frame[YEAR_COL], errors='coerce')
    years = years[years.notna() & years.ne(-1)]
    return f'{int(years.min())}–{int(years.max())}' if len(years) else 'N/A'

overview = pd.DataFrame({
    'train': {
        'rows': len(train),
        'columns': train.shape[1],
        'memory_MiB': train.memory_usage(deep=True).sum() / 1024**2,
        'duplicate_rows': int(train.duplicated().sum()),
        'unique_IDs': train[ID_COL].nunique(dropna=True),
        'missing_IDs': int(train[ID_COL].isna().sum()),
        'year_range': year_range(train),
    },
    'test': {
        'rows': len(test),
        'columns': test.shape[1],
        'memory_MiB': test.memory_usage(deep=True).sum() / 1024**2,
        'duplicate_rows': int(test.duplicated().sum()),
        'unique_IDs': test[ID_COL].nunique(dropna=True),
        'missing_IDs': int(test[ID_COL].isna().sum()),
        'year_range': year_range(test),
    },
}).T
display(overview)
display(train.head(3))

only_train = sorted(set(train.columns) - set(test.columns))
only_test = sorted(set(test.columns) - set(train.columns))
print('train only columns:', only_train)
print('test only columns :', only_test)

dtype_counts = pd.concat([
    train.dtypes.astype(str).value_counts().rename('train'),
    test.dtypes.astype(str).value_counts().rename('test'),
], axis=1).fillna(0).astype(int).reset_index(names='dtype')
dtype_long = dtype_counts.melt('dtype', var_name='split', value_name='columns')
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=dtype_long, x='dtype', y='columns', hue='split', ax=ax)
ax.set_title('Column count by dtype')
ax.tick_params(axis='x', rotation=25)
finish_figure(fig, '01_dtype_counts')

In [ ]:
profile_rows = []
for column in sorted(set(train.columns) | set(test.columns)):
    profile_rows.append({
        'column': column,
        'train_dtype': str(train[column].dtype) if column in train else None,
        'test_dtype': str(test[column].dtype) if column in test else None,
        'train_unique': train[column].nunique(dropna=True) if column in train else np.nan,
        'test_unique': test[column].nunique(dropna=True) if column in test else np.nan,
        'train_missing_rate': train[column].isna().mean() if column in train else np.nan,
        'test_missing_rate': test[column].isna().mean() if column in test else np.nan,
    })
column_profile = pd.DataFrame(profile_rows).set_index('column')
display(column_profile)

## 3. targetの分布

このコンペの評価指標はROC-AUCですが、クラス比はCVの安定性、閾値決定、subset評価の可否に影響するため確認します。

In [ ]:
target_numeric = pd.to_numeric(train[TARGET_COL], errors='coerce')
if target_numeric.isna().any() or not set(target_numeric.unique()).issubset({0, 1}):
    raise ValueError(f'{TARGET_COL} must be binary 0/1 without missing values.')
target_summary = target_numeric.value_counts().sort_index().rename('count').to_frame()
target_summary['rate'] = target_summary['count'] / len(train)
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=train, x=TARGET_COL, hue=TARGET_COL, legend=False, ax=axes[0])
axes[0].set_title('Target count')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d')
axes[1].pie(
    target_summary['count'],
    labels=[f'{TARGET_COL}={value}' for value in target_summary.index],
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('Set2', len(target_summary)),
)
axes[1].set_title('Target share')
finish_figure(fig, '02_target_balance')

## 4. 欠損値

train/testの欠損率を同じ軸で比較します。欠損率そのものが時期やsplitを表す場合があるので、補完前に差も残します。

In [ ]:
missing_profile = pd.DataFrame({
    'train': train[common_cols].isna().mean(),
    'test': test[common_cols].isna().mean(),
})
missing_profile['max_rate'] = missing_profile.max(axis=1)
missing_profile['abs_gap'] = (missing_profile['train'] - missing_profile['test']).abs()
missing_profile = missing_profile.sort_values(['max_rate', 'abs_gap'], ascending=False)
display(missing_profile.query('max_rate > 0').head(30))

plot_missing = missing_profile.query('max_rate > 0').head(30).drop(columns=['max_rate', 'abs_gap'])
if len(plot_missing):
    missing_long = plot_missing.reset_index(names='column').melt(
        'column', var_name='split', value_name='missing_rate'
    )
    fig, ax = plt.subplots(figsize=(10, max(4, 0.32 * len(plot_missing))))
    sns.barplot(data=missing_long, x='missing_rate', y='column', hue='split', ax=ax)
    ax.xaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
    ax.set_title('Missing rate: train vs test')
    finish_figure(fig, '03_missing_rates')
else:
    print('No missing values in common columns.')

## 5. 年度分布とtarget推移

時間に伴うdistribution shiftを確認します。`-1`と数値化できない年度は異常値として件数を別表示し、正常年度のグラフからだけ除外します。

In [ ]:
def valid_year_frame(frame, split):
    years = pd.to_numeric(frame[YEAR_COL], errors='coerce')
    mask = years.notna() & years.ne(-1)
    result = frame.loc[mask].copy()
    result[YEAR_COL] = years.loc[mask].astype(int)
    result['_split'] = split
    return result

train_time = valid_year_frame(train, 'train')
test_time = valid_year_frame(test, 'test')
invalid_years = pd.Series({
    'train': len(train) - len(train_time),
    'test': len(test) - len(test_time),
}, name='invalid_or_missing_year_rows')
display(invalid_years.to_frame())

year_counts = pd.concat([
    train_time.groupby(YEAR_COL).size().rename('count').reset_index().assign(split='train'),
    test_time.groupby(YEAR_COL).size().rename('count').reset_index().assign(split='test'),
], ignore_index=True)
year_target = train_time.groupby(YEAR_COL)[TARGET_COL].agg(['mean', 'count']).reset_index()
class_by_year = pd.crosstab(train_time[YEAR_COL], train_time[TARGET_COL])

fig, axes = plt.subplots(3, 1, figsize=(13, 13))
sns.lineplot(data=year_counts, x=YEAR_COL, y='count', hue='split', marker='o', ax=axes[0])
axes[0].set_title('Rows by project start year')
sns.lineplot(data=year_target, x=YEAR_COL, y='mean', marker='o', color='tab:red', ax=axes[1])
axes[1].axhline(train[TARGET_COL].mean(), color='gray', linestyle='--', label='overall train mean')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('target mean')
axes[1].set_title('Target rate by project start year')
axes[1].legend()
class_by_year.plot(kind='bar', stacked=True, ax=axes[2], color=sns.color_palette('Set2', 2))
axes[2].set_title('Target counts by project start year')
axes[2].set_ylabel('rows')
axes[2].tick_params(axis='x', rotation=45)
finish_figure(fig, '04_year_and_target')

## 6. Expanding-window CVの可視化

baselineと同じ最新3実在年度のfoldを作り、件数、target率、過去に同名projectがある割合を可視化します。

In [ ]:
folds, cv_diagnostics = make_time_series_cv(
    df=train,
    year_col=YEAR_COL,
    project_col=PROJECT_COL,
    target_col=TARGET_COL,
    n_valid_years=3,
)
display(cv_diagnostics)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
count_long = cv_diagnostics.melt(
    id_vars='validation_year',
    value_vars=['train_size', 'validation_size'],
    var_name='part',
    value_name='rows',
)
sns.barplot(data=count_long, x='validation_year', y='rows', hue='part', ax=axes[0])
axes[0].set_title('Fold sizes')
rate_long = cv_diagnostics.melt(
    id_vars='validation_year',
    value_vars=['train_target_mean', 'validation_target_mean'],
    var_name='part',
    value_name='target_rate',
)
sns.lineplot(data=rate_long, x='validation_year', y='target_rate', hue='part', marker='o', ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title('Target rate by fold')
sns.barplot(data=cv_diagnostics, x='validation_year', y='seen_project_rate', color='tab:purple', ax=axes[2])
axes[2].set_ylim(0, 1)
axes[2].set_title('Seen project rate in validation')
finish_figure(fig, '05_cv_diagnostics')

## 7. 数値特徴の分布

全数値列を複数ページに分け、train/testの分布とtarget別の分布を確認します。外れ値で中央部分が潰れないよう、histogramは1〜99 percentileへ表示範囲を限定しますが、データ自体は変更しません。

In [ ]:
numeric_summary = train[numeric_cols].apply(pd.to_numeric, errors='coerce').describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T
numeric_summary['missing_rate'] = train[numeric_cols].isna().mean()
display(numeric_summary)

def plot_numeric_pages(columns, per_page=6):
    train_plot = sampled(train)
    test_plot = sampled(test)
    for page, start in enumerate(range(0, len(columns), per_page), start=1):
        selected = columns[start:start + per_page]
        fig, axes = plt.subplots(math.ceil(len(selected) / 2), 2, figsize=(14, 4 * math.ceil(len(selected) / 2)))
        axes = np.asarray(axes).reshape(-1)
        for ax, column in zip(axes, selected):
            train_values = pd.to_numeric(train_plot[column], errors='coerce')
            test_values = pd.to_numeric(test_plot[column], errors='coerce')
            combined = pd.concat([train_values, test_values]).dropna()
            if combined.nunique() <= 1:
                ax.text(0.5, 0.5, 'constant / empty', ha='center', va='center')
                ax.set_title(column)
                continue
            lower, upper = combined.quantile([0.01, 0.99])
            sns.histplot(train_values, bins=40, stat='density', element='step', fill=False, label='train', ax=ax, binrange=(lower, upper))
            sns.histplot(test_values, bins=40, stat='density', element='step', fill=False, label='test', ax=ax, binrange=(lower, upper))
            ax.set_xlim(lower, upper)
            ax.set_title(column)
            ax.legend()
        for ax in axes[len(selected):]:
            ax.set_visible(False)
        finish_figure(fig, f'06_numeric_distribution_{page:02d}')

plot_numeric_pages(numeric_cols)

In [ ]:
def plot_numeric_target_pages(columns, per_page=6):
    train_plot = sampled(train)
    for page, start in enumerate(range(0, len(columns), per_page), start=1):
        selected = columns[start:start + per_page]
        fig, axes = plt.subplots(math.ceil(len(selected) / 2), 2, figsize=(14, 4 * math.ceil(len(selected) / 2)))
        axes = np.asarray(axes).reshape(-1)
        for ax, column in zip(axes, selected):
            plot_data = train_plot[[TARGET_COL, column]].copy()
            plot_data[column] = pd.to_numeric(plot_data[column], errors='coerce')
            plot_data = plot_data.dropna()
            if len(plot_data):
                sns.boxplot(data=plot_data, x=TARGET_COL, y=column, showfliers=False, hue=TARGET_COL, legend=False, ax=ax)
            else:
                ax.text(0.5, 0.5, 'no finite values', ha='center', va='center')
            ax.set_title(f'{column} by target (without fliers)')
        for ax in axes[len(selected):]:
            ax.set_visible(False)
        finish_figure(fig, f'07_numeric_by_target_{page:02d}')

plot_numeric_target_pages(numeric_cols)

corr_data = train[numeric_cols + [TARGET_COL]].apply(pd.to_numeric, errors='coerce')
nonconstant = [column for column in corr_data if corr_data[column].nunique(dropna=True) > 1]
if len(nonconstant) >= 2:
    corr = corr_data[nonconstant].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(max(8, 0.65 * len(corr)), max(6, 0.55 * len(corr))))
    sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title('Spearman correlation (numeric + target)')
    finish_figure(fig, '08_numeric_correlation')

## 8. カテゴリ特徴

各列について上位カテゴリのtrain/test構成比と、十分な件数があるカテゴリのtarget率を表示します。カテゴリ名は文字列化し、欠損を`<MISSING>`として可視化します。

In [ ]:
def category_series(frame, column):
    return frame[column].fillna('<MISSING>').astype(str)

def plot_category(column):
    train_cat = category_series(train, column)
    test_cat = category_series(test, column)
    top = pd.concat([train_cat, test_cat]).value_counts().head(TOP_N).index
    distribution_parts = []
    for split, values in [('train', train_cat), ('test', test_cat)]:
        share = values.value_counts(normalize=True).reindex(top, fill_value=0)
        distribution_parts.append(pd.DataFrame({'category': top, 'share': share.values, 'split': split}))
    distribution = pd.concat(distribution_parts, ignore_index=True)
    target_stats = pd.DataFrame({'category': train_cat, 'target': train[TARGET_COL]}).groupby('category')['target'].agg(['mean', 'count'])
    target_stats = target_stats.query('count >= @MIN_CATEGORY_COUNT').sort_values('count', ascending=False).head(TOP_N).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, 0.38 * len(top))))
    sns.barplot(data=distribution, x='share', y='category', hue='split', ax=axes[0])
    axes[0].xaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
    axes[0].set_title(f'{column}: top category share')
    if len(target_stats):
        sns.barplot(data=target_stats, x='mean', y='category', color='tab:red', ax=axes[1])
        axes[1].axvline(train[TARGET_COL].mean(), color='gray', linestyle='--', label='overall')
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('target rate')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, f'No category with n >= {MIN_CATEGORY_COUNT}', ha='center')
    axes[1].set_title(f'{column}: target rate (frequent categories)')
    finish_figure(fig, f'09_category_{column}')

category_overview = pd.DataFrame({
    'train_unique': train[categorical_cols].nunique(dropna=True),
    'test_unique': test[categorical_cols].nunique(dropna=True),
    'train_missing_rate': train[categorical_cols].isna().mean(),
    'test_missing_rate': test[categorical_cols].isna().mean(),
}) if categorical_cols else pd.DataFrame()
display(category_overview)
for column in categorical_cols:
    plot_category(column)

## 9. 日本語テキストの長さ

本文そのものを表示しすぎず、文字数・空文字率・target差・年度変化を確認します。文字数はTF-IDFやEmbedding APIの計算量、truncate方針にも関係します。

In [ ]:
def normalized_text_length(series):
    text = series.fillna('').astype(str).str.strip()
    return text.str.len()

text_length_summary = []
for column in text_cols:
    for split, frame in [('train', train), ('test', test)]:
        length = normalized_text_length(frame[column])
        text_length_summary.append({
            'column': column,
            'split': split,
            'empty_rate': float(length.eq(0).mean()),
            'median_chars': float(length.median()),
            'p95_chars': float(length.quantile(0.95)),
            'max_chars': int(length.max()),
        })
display(pd.DataFrame(text_length_summary))

for column in text_cols:
    train_length = normalized_text_length(train[column])
    test_length = normalized_text_length(test[column])
    train_plot = sampled(pd.DataFrame({'length': train_length, TARGET_COL: train[TARGET_COL], YEAR_COL: train[YEAR_COL]}))
    test_plot = sampled(pd.DataFrame({'length': test_length}))
    upper = max(1, float(pd.concat([train_length, test_length]).quantile(0.99)))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    sns.histplot(train_plot['length'], bins=50, stat='density', element='step', fill=False, label='train', binrange=(0, upper), ax=axes[0])
    sns.histplot(test_plot['length'], bins=50, stat='density', element='step', fill=False, label='test', binrange=(0, upper), ax=axes[0])
    axes[0].set_xlim(0, upper)
    axes[0].set_title('Character length distribution')
    axes[0].legend()
    sns.boxplot(data=train_plot, x=TARGET_COL, y='length', showfliers=False, hue=TARGET_COL, legend=False, ax=axes[1])
    axes[1].set_title('Length by target (without fliers)')
    length_year = pd.DataFrame({YEAR_COL: pd.to_numeric(train[YEAR_COL], errors='coerce'), 'length': train_length})
    length_year = length_year[length_year[YEAR_COL].notna() & length_year[YEAR_COL].ne(-1)].groupby(YEAR_COL)['length'].median().reset_index()
    sns.lineplot(data=length_year, x=YEAR_COL, y='length', marker='o', ax=axes[2])
    axes[2].set_title('Median length by start year')
    fig.suptitle(column, y=1.03, fontsize=14)
    finish_figure(fig, f'10_text_length_{column}')

## 10. `project_name`の重複・train/test overlap

同じ`project_name`をGroupKFoldで隔離する設計にはしません。本番testにもtrainで過去に登場した名称があり得るため、重複とoverlapはデータ構造として把握します。

In [ ]:
train_names = train[PROJECT_COL].dropna().astype(str)
test_names = test[PROJECT_COL].dropna().astype(str)
known_names = set(train_names.unique())
test_seen = test[PROJECT_COL].notna() & test[PROJECT_COL].astype(str).isin(known_names)
name_target_nunique = train.dropna(subset=[PROJECT_COL]).groupby(PROJECT_COL)[TARGET_COL].nunique()
project_summary = pd.Series({
    'train_rows_with_duplicated_name': int(train[PROJECT_COL].duplicated(keep=False).sum()),
    'train_duplicated_name_rate': float(train[PROJECT_COL].duplicated(keep=False).mean()),
    'train_unique_names': train[PROJECT_COL].nunique(dropna=True),
    'test_unique_names': test[PROJECT_COL].nunique(dropna=True),
    'test_names_seen_in_train': int(test_seen.sum()),
    'test_seen_name_rate': float(test_seen.mean()),
    'train_names_with_conflicting_targets': int(name_target_nunique.gt(1).sum()),
})
display(project_summary.to_frame('value'))

name_counts = train_names.value_counts()
frequency_of_frequency = name_counts.value_counts().sort_index().rename_axis('rows_per_name').rename('number_of_names').reset_index()
top_names = name_counts.head(TOP_N).sort_values().rename_axis(PROJECT_COL).rename('rows').reset_index()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=frequency_of_frequency.head(30), x='rows_per_name', y='number_of_names', color='tab:blue', ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('How many times each project name appears')
sns.barplot(data=top_names, x='rows', y=PROJECT_COL, color='tab:green', ax=axes[1])
axes[1].set_title(f'Top {TOP_N} repeated project names')
finish_figure(fig, '11_project_name_duplicates')

## 11. 単変量のtrain/test差

数値列はKolmogorov–Smirnov統計量、カテゴリ列はtotal variation distanceで単変量の差を順位付けします。これは軽量なQuick EDAであり、モデルを使うadversarial validationではありません。値が大きい列は年度推移や欠損率も併せて確認します。

In [ ]:
numeric_drift_rows = []
for column in numeric_cols:
    train_values = pd.to_numeric(train[column], errors='coerce').dropna()
    test_values = pd.to_numeric(test[column], errors='coerce').dropna()
    if len(train_values) and len(test_values):
        statistic, pvalue = ks_2samp(train_values, test_values)
        numeric_drift_rows.append({'column': column, 'ks_statistic': statistic, 'pvalue': pvalue})
numeric_drift = pd.DataFrame(numeric_drift_rows).sort_values('ks_statistic', ascending=False) if numeric_drift_rows else pd.DataFrame()

categorical_drift_rows = []
for column in categorical_cols:
    train_share = category_series(train, column).value_counts(normalize=True)
    test_share = category_series(test, column).value_counts(normalize=True)
    categories = train_share.index.union(test_share.index)
    tv_distance = 0.5 * (train_share.reindex(categories, fill_value=0) - test_share.reindex(categories, fill_value=0)).abs().sum()
    categorical_drift_rows.append({'column': column, 'total_variation': tv_distance})
categorical_drift = pd.DataFrame(categorical_drift_rows).sort_values('total_variation', ascending=False) if categorical_drift_rows else pd.DataFrame()
display(numeric_drift)
display(categorical_drift)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
if len(numeric_drift):
    sns.barplot(data=numeric_drift.head(20), x='ks_statistic', y='column', color='tab:blue', ax=axes[0])
axes[0].set_title('Numeric train/test drift: KS statistic')
if len(categorical_drift):
    sns.barplot(data=categorical_drift.head(20), x='total_variation', y='column', color='tab:orange', ax=axes[1])
axes[1].set_title('Categorical train/test drift: total variation')
finish_figure(fig, '12_univariate_train_test_drift')

## 12. baseline前チェックリスト

次を確認したら、`01_time_series_cv.ipynb`、`02_char_tfidf_logreg.ipynb`、`04_embedding_models.ipynb`へ進みます。

- targetが0/1で、極端なクラス偏りがないか
- trainの最新年度よりtestが未来側か
- `project_start_year == -1`の件数と他の年度列で補える可能性
- 欠損率やカテゴリ構成がtrain/testで大きく変わる列
- 予算などの外れ値に`log1p`が必要か
- textの空文字、極端に長い行、年度による文字数変化
- `project_name`重複、target矛盾、testでのseen率
- 最新foldの件数、target率、seen/unseen率

必要なら`SAVE_FIGURES=True`にして、図を`data/eda_figures/`へ保存します。